In [1]:

import pandas as pd
from itertools import combinations



file_path = "/content/Market_Basket_Optimisation.csv"

df = pd.read_csv(
    file_path,
    header=None
)

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (7501, 20)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:


transactions = []

for i in range(len(df)):

    items = set()

    for item in df.iloc[i]:

        if pd.notna(item):

            items.add(str(item).strip())

    transactions.append(items)

print("Total Transactions:", len(transactions))

print("\nFirst 5 Transactions:")

for transaction in transactions[:5]:

    print(transaction)

Total Transactions: 7501

First 5 Transactions:
{'tomato juice', 'frozen smoothie', 'salmon', 'whole weat flour', 'green grapes', 'salad', 'mineral water', 'olive oil', 'energy drink', 'avocado', 'shrimp', 'green tea', 'yams', 'honey', 'almonds', 'spinach', 'vegetables mix', 'antioxydant juice', 'cottage cheese', 'low fat yogurt'}
{'burgers', 'meatballs', 'eggs'}
{'chutney'}
{'avocado', 'turkey'}
{'mineral water', 'green tea', 'milk', 'energy bar', 'whole wheat rice'}


In [3]:
min_support = 0.01

total_transactions = len(transactions)

min_support_count = int(

    min_support * total_transactions

)

print("Minimum Support:", min_support)

print("Minimum Support Count:",

      min_support_count)

Minimum Support: 0.01
Minimum Support Count: 75


In [4]:
item_count = {}

for transaction in transactions:

    for item in transaction:

        if item not in item_count:

            item_count[item] = 0

        item_count[item] += 1

frequent_itemsets = {}

for item, count in item_count.items():

    if count >= min_support_count:

        frequent_itemsets[

            frozenset([item])

        ] = count

print(

    "Frequent 1-Itemsets:",

    len(frequent_itemsets)

)

Frequent 1-Itemsets: 75


In [5]:


def generate_candidates(previous_itemsets, k):

    candidates = set()

    previous_list = list(previous_itemsets)

    for i in range(len(previous_list)):

        for j in range(i + 1, len(previous_list)):

            union = (

                previous_list[i]

                | previous_list[j]

            )

            if len(union) == k:

                candidates.add(union)

    return candidates

In [6]:


def count_support(

    candidates,

    transactions

):

    counts = {}

    for candidate in candidates:

        count = 0

        for transaction in transactions:

            if candidate.issubset(transaction):

                count += 1

        counts[candidate] = count

    return counts

In [7]:
all_frequent_itemsets = {}

# Add 1-itemsets

all_frequent_itemsets.update(

    frequent_itemsets

)

current_itemsets = frequent_itemsets

k = 2

while current_itemsets:

    # Generate candidates

    candidates = generate_candidates(

        current_itemsets.keys(),

        k

    )

    if not candidates:

        break

    # Count support

    candidate_counts = count_support(

        candidates,

        transactions

    )

    # Keep frequent candidates

    new_frequent_itemsets = {}

    for itemset, count in candidate_counts.items():

        if count >= min_support_count:

            new_frequent_itemsets[

                itemset

            ] = count

    if not new_frequent_itemsets:

        break

    all_frequent_itemsets.update(

        new_frequent_itemsets

    )

    current_itemsets = new_frequent_itemsets

    k += 1

print(

    "Total Frequent Itemsets:",

    len(all_frequent_itemsets)

)

Total Frequent Itemsets: 260


In [8]:
results = []

for itemset, count in all_frequent_itemsets.items():

    support = count / total_transactions

    results.append({

        "Itemset": ", ".join(

            sorted(itemset)

        ),

        "Support": support,

        "Count": count

    })

result_df = pd.DataFrame(results)

result_df = result_df.sort_values(

    by="Support",

    ascending=False

)

display(

    result_df.head(20)

)

,Itemset,Support,Count
3,mineral water,0.238368,1788
17,eggs,0.179709,1348
26,spaghetti,0.174110,1306
22,french fries,0.170911,1282
31,chocolate,0.163845,1229
8,green tea,0.132116,991
19,milk,0.129583,972
51,ground beef,0.098254,737
27,frozen vegetables,0.095321,715
45,pancakes,0.095054,713


In [9]:


rules = []

min_confidence = 0.30

for itemset, itemset_count in all_frequent_itemsets.items():

    if len(itemset) < 2:
        continue

    itemset_support = (
        itemset_count /
        total_transactions
    )


    for i in range(1, len(itemset)):

        for antecedent_tuple in combinations(
            itemset,
            i
        ):

            antecedent = frozenset(
                antecedent_tuple
            )

            consequent = (
                itemset - antecedent
            )

            if antecedent not in all_frequent_itemsets:
                continue

            antecedent_count = (
                all_frequent_itemsets[
                    antecedent
                ]
            )

            confidence = (
                itemset_count /
                antecedent_count
            )

            if confidence >= min_confidence:

                antecedent_support = (
                    antecedent_count /
                    total_transactions
                )

                consequent_count = 0

                for transaction in transactions:

                    if consequent.issubset(transaction):
                        consequent_count += 1

                consequent_support = (
                    consequent_count /
                    total_transactions
                )

                lift = (
                    confidence /
                    consequent_support
                )

                rules.append({

                    "Antecedent":
                    ", ".join(
                        sorted(antecedent)
                    ),

                    "Consequent":
                    ", ".join(
                        sorted(consequent)
                    ),

                    "Support":
                    itemset_support,

                    "Confidence":
                    confidence,

                    "Lift":
                    lift
                })

In [10]:


rules_df = pd.DataFrame(rules)

rules_df = rules_df.sort_values(
    by="Lift",
    ascending=False
)

rules_df["Support"] = rules_df[
    "Support"
].round(4)

rules_df["Confidence"] = rules_df[
    "Confidence"
].round(4)

rules_df["Lift"] = rules_df[
    "Lift"
].round(4)

print("Top Association Rules:")

display(
    rules_df.head(20)
)

Top Association Rules:


,Antecedent,Consequent,Support,Confidence,Lift
14,herb & pepper,ground beef,0.0160,0.3235,3.2920
43,"ground beef, mineral water",spaghetti,0.0171,0.4169,2.3947
56,"frozen vegetables, mineral water",milk,0.0111,0.3097,2.3900
27,soup,milk,0.0152,0.3008,2.3212
7,ground beef,spaghetti,0.0392,0.3989,2.2912
62,"mineral water, olive oil",spaghetti,0.0103,0.3720,2.1365
39,"eggs, ground beef",mineral water,0.0101,0.5067,2.1256
58,"ground beef, milk",mineral water,0.0111,0.5030,2.1103
6,red wine,spaghetti,0.0103,0.3649,2.0960
4,olive oil,spaghetti,0.0229,0.3482,1.9998
